# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


####  Run this cell to set up and start your interactive session.


#### Example: Create a DynamicFrame from a table in the AWS Glue Data Catalog and display its schema


#### Example: Convert the DynamicFrame to a Spark DataFrame and display a sample of the data


#### Example: Visualize data with matplotlib


#### Example: Write the data in the DynamicFrame to a location in Amazon S3 and a table for it in the AWS Glue Data Catalog


In [1]:
import sys
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from pyspark.sql.functions import upper, col, trim

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: 7986d086-1f83-4280-972f-0d3f39d271a2
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 7986d086-1f83-4280-972f-0d3f39d271a2 to get into ready status...
Session 7986d086-1f83-4280-972f-0d3f39d271a2 has been created.



In [2]:
# 1. INITIALIZE: handshaking with the Spark cluster
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

In [3]:
# 2. READ: Load the raw CSV from your S3 bucket
# We use the path we verified in Project 1
input_path = "s3://zs-nivedh-raw-data-2026/Sales_Dataset_2024.csv"
df = spark.read.option("header", "true").csv(input_path)

In [4]:
# 3. TRANSFORM: The Cleaning Logic
# - We fix "south" vs "South" by making everything UPPERCASE
# - We trim any hidden spaces
# - We cast numbers to the correct types for better analytics
df_cleaned = df \
    .filter(col("Region").isNotNull()) \
    .withColumn("Region", upper(trim(col("Region")))) \
    .withColumn("Units_Sold", col("Units_Sold").cast("int")) \
    .withColumn("Revenue", col("Revenue").cast("double")) \
    .withColumn("Profit", col("Profit").cast("double"))

In [5]:
# 4. PREVIEW: Show the first 10 rows of the cleaned data
print("PREVIEW OF CLEANED DATA:")
df_cleaned.show(10)

PREVIEW OF CLEANED DATA:
+-------------------+------+----------+-----------+----------+----------+-----------+-------+-----------+-----------+
|               Date|Region|   Product|Salesperson|Units_Sold|Unit_Price|   Category|Revenue|       Cost|     Profit|
+-------------------+------+----------+-----------+----------+----------+-----------+-------+-----------+-----------+
|2024-04-12 00:00:00| NORTH|Smartwatch|     Hannah|        15|      1224|Accessories|18360.0|16451.63426|1908.365742|
|2024-12-14 00:00:00| NORTH|   Monitor|        Eva|         5|      1321|     Office| 6605.0|4457.351727|2147.648273|
|2024-09-27 00:00:00| NORTH|    Mobile|        Bob|        11|       912|Electronics|10032.0|6563.644126|3468.355874|
|2024-04-16 00:00:00|  WEST|   Monitor|    Charlie|        18|       325|     Office| 5850.0|4320.807092|1529.192908|
|2024-03-12 00:00:00|  WEST|Headphones|        Eva|        13|      1042|Accessories|13546.0|8270.122666|5275.877334|
|2024-07-07 00:00:00|  WEST|   

In [7]:
# 5. WRITE: Save as PARQUET (Big Data Standard)
# This creates a folder in your bucket with the optimized files
output_path = "s3://zs-nivedh-raw-data-2026/cleaned_parquet_data/"
df_cleaned.write.mode("overwrite").parquet(output_path)

print("SUCCESS: Data cleaned and saved to Parquet!")

SUCCESS: Data cleaned and saved to Parquet!
